# Data example

A walk-through of loading and inspecting a curve dataset from `inputs/`. Pick a `CURVE`, load its daily par-rate series, and look at coverage, summary statistics, and a few plots — the same data that feeds covariance estimation.

In [ ]:
import sys
from pathlib import Path

# Make the project package importable regardless of where Jupyter was launched.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_rates
from src.tenor_graph import tenor_to_years

INPUTS = ROOT / "inputs"
print("available curves:", [p.name for p in sorted(INPUTS.iterdir()) if p.is_dir()])

## 1. Load a curve

Each curve folder holds `curve_data.csv` (daily par rates in long form) and `tenor_structure.json`. Set `CURVE` to one of the names above. The CSV carries a few constant metadata columns alongside the `Date / Tenor / ParRate` series.

In [ ]:
CURVE = "CAD"
base  = INPUTS / CURVE

raw = pd.read_csv(base / "curve_data.csv", parse_dates=["Date"])
print("raw shape:", raw.shape)
print(raw[["Currency", "DiscCurve", "ForecastCurve"]].iloc[0].to_string())
raw.head()

## 2. Pivot to a wide rate table

`load_rates` pivots the long CSV into a `Date x Tenor` table with tenors sorted by maturity — exactly the shape the estimators consume.

In [ ]:
rates = load_rates(base / "curve_data.csv")
print(f"{rates.shape[0]} dates x {rates.shape[1]} tenors")
print("date range:", rates.index.min().date(), "->", rates.index.max().date())
print("tenors:", list(rates.columns))
rates.head()

## 3. Coverage & summary stats

Check for missing observations (gaps where a tenor wasn't quoted), then look at the per-tenor level distribution.

In [ ]:
missing = rates.isna().sum()
print("missing values per tenor:")
print(missing[missing > 0].to_string() if missing.any() else "  none")

rates.describe().round(3).T

## 4. Par-rate history

Levels through time for every tenor — the whole term structure moving day to day.

In [ ]:
ax = rates.plot(figsize=(11, 5), lw=1)
ax.set(xlabel="Date", ylabel="Par rate (%)", title=f"{CURVE} par rates")
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()

## 5. Snapshot yield curves

A handful of evenly spaced dates plotted against maturity show the curve shape (upward-sloping / inverted) on each day.

In [ ]:
years  = [tenor_to_years(t) for t in rates.columns]
sample = rates.iloc[:: max(len(rates) // 5, 1)]   # ~5 evenly spaced dates

fig, ax = plt.subplots(figsize=(9, 5))
for date, row in sample.iterrows():
    ax.plot(years, row.values, marker="o", ms=3, label=date.date())
ax.set(xlabel="Maturity (years)", ylabel="Par rate (%)", title=f"{CURVE} curve snapshots")
ax.legend(title="Date", fontsize=8)
plt.tight_layout()

## 6. Daily changes

Covariance estimation runs on daily *changes*, not levels. Here are their scale per tenor and the pairwise correlation across the curve — neighbouring tenors move almost in lockstep, which is what dimension reduction exploits.

In [ ]:
changes = rates.diff().dropna()
print(f"{changes.shape[0]} daily changes")
display(changes.std().round(4).rename("daily vol").to_frame().T)

corr = changes.corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr)), corr.index, fontsize=8)
ax.set_title(f"{CURVE} daily-change correlation")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()